# Wildfire GRPO Training (Hugging Face)

Use this notebook only for GRPO training on the Hugging Face GPU runtime.

Cell order:
1. Run Cell 1 first each session (loads env vars + BASE).
2. Run Cell 2 to clone/update and install dependencies.
3. Run Cell 3 for OpenEnv/training preflight.
4. Run Cell 4 for the resume-safe GRPO training run.

After training finishes, use `notebooks/wildfire_http_eval_hf.ipynb` for HTTP baseline/trained evaluation and final artifacts.

In [ ]:
# Cell 1 (run first each session): env vars + constants (HF runtime)
import os

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
HF_TOKEN = os.environ.get("HF_TOKEN")

if not GITHUB_TOKEN:
    raise RuntimeError("Set GITHUB_TOKEN in the Hugging Face Space/Notebook secrets.")
if not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN in the Hugging Face Space/Notebook secrets.")

os.environ["HF_TOKEN"] = HF_TOKEN

REPO = "jhawaritvik/Scaler-hackathon"
REPO_URL = f"https://github.com/{REPO}.git"
BASE = "/home/user/app/wildfire-env"

print("Configured BASE:", BASE)
print("Tokens available: GITHUB_TOKEN + HF_TOKEN")

In [ ]:
# Cell 2: clone/update + install deps (safe to re-run)
import os
import glob
import subprocess
import sys

if not os.path.exists(BASE):
    subprocess.run(["git", "clone", f"https://{GITHUB_TOKEN}@github.com/{REPO}.git", BASE], check=True)

os.chdir(BASE)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", "main"], check=True)
subprocess.run(["git", "pull", "--rebase", "origin", "main"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[train]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "unsloth", "unsloth-zoo", "bitsandbytes", "xgrammar", "transformers", "trl"], check=True)

nvidia_lib_dirs = glob.glob("/usr/local/lib/python*/dist-packages/nvidia/*/lib")
if nvidia_lib_dirs:
    os.environ["LD_LIBRARY_PATH"] = ":".join(nvidia_lib_dirs) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
    print("LD_LIBRARY_PATH updated with NVIDIA libs")

print("Install complete. cwd:", os.getcwd())

In [ ]:
# Cell 3: preflight for training stack
import subprocess
import sys

subprocess.run(["openenv", "validate", "."], check=True)
subprocess.run([sys.executable, "smoke_test.py"], check=True)
print("Preflight passed")

In [ ]:
# Cell 4: full GRPO training (resume-safe)
import subprocess
import sys

subprocess.run([sys.executable, "train_grpo.py"], check=True)
print("Training run finished")

In [ ]:
# Stop Here

Training is complete after Cell 4. Run HTTP evaluation and final artifact generation from `notebooks/wildfire_http_eval_hf.ipynb`.